# ClimateTwin India • Training data and model results

Explore the synthetic daily records used for ClimateTwin's monthly temperature and rainfall projections. This notebook reads the complete daily table exported from the project's SQLite database and reproduces the evaluation used by the dashboard.

**Run it** by extracting the ZIP, opening this notebook in Jupyter or VS Code, and choosing **Run All** with a Python kernel that has pandas, NumPy and scikit-learn installed. In Google Colab, open this notebook and upload `climate_daily.csv` through the Files panel before running the cells. The saved outputs are available to read immediately.

## 1. Inspect the daily data

Each row describes one region on one day. Temperatures are in °C and rainfall is in mm per day. Latitude and longitude identify the region's representative location. Blank measurements remain missing. The live weather panel uses a separate data source.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

data_file = Path("climate_daily.csv")
daily = pd.read_csv(data_file, parse_dates=["date"])
targets = ["min_temp", "max_temp", "rainfall"]
print(f"{len(daily):,} daily rows | {daily['state'].nunique()} states and union territories")
print(f"{daily['date'].min():%Y-%m-%d} through {daily['date'].max():%Y-%m-%d}")
print("Missing values:", daily[targets].isna().sum().to_dict())
daily.loc[daily["state"].eq("Delhi"), ["date", "state", *targets]].head()

223,524 daily rows | 36 states and union territories
2010-01-01 through 2026-12-31
Missing values: {'min_temp': 117, 'max_temp': 103, 'rainfall': 115}


,date,state,min_temp,max_temp,rainfall
49672,2010-01-01,Delhi,8.9,25.9,0.0
49673,2010-01-02,Delhi,9.8,24.1,0.0
49674,2010-01-03,Delhi,11.5,25.7,0.0
49675,2010-01-04,Delhi,9.7,27.9,0.0
49676,2010-01-05,Delhi,12.5,24.8,0.0


## 2. Prepare one example per region and month

Average the daily lows, average the daily highs, and add the rainfall. A month qualifies only when every date has all three measurements. A measured zero counts as a dry day.

In [2]:
if daily.duplicated(["state", "date"]).any():
    raise ValueError("Duplicate region/date in the source data.")

groups = daily.assign(period=daily["date"].dt.to_period("M")).groupby(["state", "period"])
days_expected = groups.size().index.get_level_values("period").days_in_month
complete = groups.size().eq(days_expected) & groups[targets].count().eq(days_expected, axis=0).all(axis=1)
monthly = groups.agg(min_temp=("min_temp", "mean"), max_temp=("max_temp", "mean"),
                     rainfall=("rainfall", "sum")).loc[complete].reset_index()
print(f"{len(monthly):,} complete region-months; {(~complete).sum():,} incomplete groups excluded.")
monthly.loc[monthly["state"].eq("Delhi") & monthly["period"].eq("2024-07")].round(dict.fromkeys(targets, 2))

7,018 complete region-months; 326 incomplete groups excluded.


,state,period,min_temp,max_temp,rainfall
1731,Delhi,2024-07,26.19,39.42,404.8


## 3. Fit the regressions and test them on later months

Fit one linear regression per region using the complete months in 2010–2024. The inputs are elapsed years and indicators for February through December, with January as the reference. Hold out 2025–2026 for testing. For comparison, the seasonal baseline uses the training average for that region and calendar month.

In [3]:
base_year = monthly["period"].min().year
test_start = monthly["period"].max() - 23

def features(periods):
    months = pd.PeriodIndex(periods, freq="M")
    elapsed = months.year - base_year + (months.month - 1) / 12
    return np.column_stack([elapsed, *[months.month == number for number in range(2, 13)]])

def predict_months(estimator, periods):
    values = estimator.predict(features(periods))
    values[:, :2] = np.sort(values[:, :2], axis=1)
    values[:, 2] = np.maximum(0, values[:, 2])
    return values

actual, predicted, seasonal = [], [], []
for region, rows in monthly.groupby("state"):
    train = rows.loc[rows["period"] < test_start]
    test = rows.loc[rows["period"] >= test_start]
    estimator = LinearRegression().fit(features(train["period"]), train[targets].to_numpy())
    predicted.append(predict_months(estimator, test["period"]))
    actual.append(test[targets].to_numpy())
    calendar_means = train.groupby(train["period"].dt.month)[targets].mean()
    seasonal.append(calendar_means.loc[test["period"].dt.month].to_numpy())

print(f"Held-out test: {test_start} through {monthly['period'].max()}")
print(f"{sum(len(rows) for rows in actual)} region-months across {len(actual)} regions")

Held-out test: 2025-01 through 2026-12
828 region-months across 36 regions


## 4. Check prediction errors

Mean absolute error (MAE) is the average size of a prediction error. Smaller is better. Mean temperature is estimated halfway between the monthly average low and high. These scores measure performance on generated data.

In [4]:
def include_mean(batches):
    values = np.concatenate(batches)
    return np.column_stack([values, values[:, :2].mean(axis=1)])

observed = include_mean(actual)
scores = pd.DataFrame(
    {
        "Regression MAE": np.abs(include_mean(predicted) - observed).mean(axis=0),
        "Seasonal baseline MAE": np.abs(include_mean(seasonal) - observed).mean(axis=0),
    },
    index=["Average daily low (°C)", "Average daily high (°C)", "Monthly rainfall (mm)", "Mean temperature (°C)"],
)
scores.round(3)

,Regression MAE,Seasonal baseline MAE
Average daily low (°C),0.178,0.228
Average daily high (°C),0.231,0.270
Monthly rainfall (mm),40.753,39.899
Mean temperature (°C),0.142,0.205


## 5. Try a 2027 projection

After evaluating on held-out months, refit the selected region using all its complete records through 2026. Change the region name or select another month in 2027 and rerun this cell to explore the result.

In [5]:
region = "Delhi"
month = "2027-07"
period = pd.Period(month, freq="M")
rows = monthly.loc[monthly["state"].eq(region)]
if rows.empty:
    raise ValueError("Choose a region listed in daily['state'].unique().")
if period not in pd.period_range(monthly["period"].max() + 1, periods=12, freq="M"):
    raise ValueError("Choose a month within the 12-month projection window.")

final_model = LinearRegression().fit(features(rows["period"]), rows[targets].to_numpy())
low, high, rainfall = predict_months(final_model, [period])[0]
example = pd.DataFrame([{"Region": region, "Month": month, "Average low (°C)": low,
                         "Average high (°C)": high, "Mean (°C)": (low + high) / 2,
                         "Monthly rainfall (mm)": rainfall}])
example.round(2)

,Region,Month,Average low (°C),Average high (°C),Mean (°C),Monthly rainfall (mm)
0,Delhi,2027-07,26.37,40.02,33.2,341.48
